In [0]:
dbutils.widgets.text("catalog", "banking")
dbutils.widgets.text("schema_landing", "landing")
dbutils.widgets.text("schema_bronze", "bronze")
dbutils.widgets.text("schema_silver", "silver")
dbutils.widgets.text("schema_gold", "gold")

catalog = dbutils.widgets.get("catalog")
schema_landing = dbutils.widgets.get("schema_landing")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")

silver_transactions_table = f"{catalog}.{schema_silver}.silver_transactions"
silver_payement_gateway_table = f"{catalog}.{schema_silver}.silver_payement_gateway"
gold_table_full = f"{catalog}.{schema_gold}.gold_transaction_channel_summary"

In [0]:
from pyspark.sql import functions as F

gold_df = (
    spark.table(silver_transactions_table)
    .join(
        spark.table(silver_payement_gateway_table),
        F.col("silver_transactions.txn_id") == F.col("silver_payement_gateway.txn_id"),
        "inner"
    )
    .groupBy(
        F.to_date("silver_transactions.txn_timestamp").alias("txn_date"),
        F.col("silver_payement_gateway.gateway_name"),
        F.col("silver_payement_gateway.device_type")
    )
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("silver_payement_gateway.gateway_status") == "SUCCESS", 1).otherwise(0)).alias("successful_transactions"),
        F.sum(F.when(F.col("silver_payement_gateway.gateway_status") == "FAILED", 1).otherwise(0)).alias("failed_transactions"),
        F.avg(F.col("silver_payement_gateway.processing_time_ms").cast("double")).alias("avg_processing_time_ms")
    )
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table_full)

In [0]:

count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM banking.gold.gold_transaction_channel_summary
""").collect()[0]["cnt"]

dbutils.notebook.exit(str(count))